# 🚦 Smart City Traffic Demand Prediction

---

## Objective

Develop a machine learning system capable of **accurately predicting traffic demand** (vehicles/hour) using historical traffic, road infrastructure, weather, and geolocation information.

## Models Used

| Model | Type | Key Strength |
|---|---|---|
| **Random Forest** | Bagging ensemble | Robust, low variance |
| **XGBoost** | Gradient boosting | High accuracy, regularisation |
| **LightGBM** | Leaf-wise gradient boosting | Speed + accuracy |
| **Weighted Ensemble** | Model averaging | Best overall generalisation |

## Evaluation Metrics

| Metric | Description |
|---|---|
| **R²** | Proportion of variance explained (higher = better) |
| **RMSE** | Root Mean Squared Error — penalises large errors |
| **MAE** | Mean Absolute Error — average magnitude of errors |
| **MAPE** | Mean Absolute Percentage Error — relative error (%) |

## Notebook Structure

1. Project Introduction  
2. Business Problem  
3. Dataset Description  
4. Import Libraries  
5. Load Dataset  
6. Exploratory Data Analysis  
7. Data Cleaning  
8. Feature Engineering  
9. Data Splitting  
10. Model Training  
11. Hyperparameter Tuning  
12. Ensemble Learning  
13. Model Evaluation  
14. Residual Analysis  
15. Feature Importance & SHAP  
16. Conclusion  
17. Future Improvements  

---
## 3. Import Libraries

In [ ]:
import os
import sys
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Add project root to path
sys.path.append('..')
from src.preprocessing import preprocess_pipeline
from src.features import FeatureEngineer, get_model_features_list
from src.ensemble import WeightedEnsemble

warnings.filterwarnings('ignore')

# ── Global Plot Aesthetics ──────────────────────────────────────────────────
PALETTE   = "viridis"
PRIMARY   = "#4C72B0"
SECONDARY = "#DD8452"
ACCENT    = "#55A868"
RED       = "#C44E52"

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'axes.labelsize'   : 13,
    'axes.titlesize'   : 15,
    'axes.titleweight' : 'bold',
    'figure.titlesize' : 18,
    'figure.titleweight': 'bold',
    'xtick.labelsize'  : 11,
    'ytick.labelsize'  : 11,
})

# ── Paths ───────────────────────────────────────────────────────────────────
DATA_PATH    = "../data/raw/smart_city_traffic_data.csv"
MODELS_DIR   = "../models"
REPORTS_DIR  = "../reports"
os.makedirs(os.path.join(REPORTS_DIR, 'figures'), exist_ok=True)

print("✅ Libraries imported successfully.")
print(f"   NumPy  : {np.__version__}")
print(f"   Pandas : {pd.__version__}")

In [ ]:
import os
import pandas as pd
os.chdir('..') # move to project root
# Load split data (saved by train.py or step 2)
train_df = pd.read_csv('data/processed/train.csv')
val_df = pd.read_csv('data/processed/val.csv')
test_df = pd.read_csv('data/processed/test.csv')

train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
val_df['timestamp'] = pd.to_datetime(val_df['timestamp'])
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])


---
## 9. Model Training

We train three complementary base models, then combine them into a weighted ensemble.

### 9.1 Random Forest

**How it works:** Random Forest builds many independent decision trees on random subsets of the training data (bagging) and random subsets of features (feature randomness). Predictions are averaged across all trees.

**Advantages:**
- Robust to overfitting due to averaging
- Handles mixed feature types well
- Provides natural feature importance via impurity reduction

**Disadvantages:**
- Slower inference than boosting models
- Memory-intensive for large forests
- Tends to underperform on sparse/tabular data versus gradient boosting

In [ ]:
# Load pre-fitted FeatureEngineer and transform splits
fe = FeatureEngineer.load('models/feature_engineer.pkl')

features = get_model_features_list()
target   = 'traffic_demand'

train_trans = fe.transform(train_df)
val_trans   = fe.transform(val_df)
test_trans  = fe.transform(test_df)

X_train, y_train = train_trans[features], train_trans[target]
X_val,   y_val   = val_trans[features],   val_trans[target]
X_test,  y_test  = test_trans[features],  test_trans[target]

print(f"Feature matrix shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")
print(f"Total features: {len(features)}")

In [ ]:
# Load pre-trained models (trained via train.py)
rf_model   = joblib.load('models/rf_model.pkl')
xgb_model  = joblib.load('models/xgb_model.pkl')
lgbm_model = joblib.load('models/lgbm_model.pkl')

print("✅ Loaded pre-trained models from disk.")
print(f"   Random Forest : {rf_model.n_estimators} trees, max_depth={rf_model.max_depth}")
print(f"   XGBoost       : {xgb_model.n_estimators} rounds, max_depth={xgb_model.max_depth}")
print(f"   LightGBM      : {lgbm_model.n_estimators} rounds, num_leaves={lgbm_model.num_leaves}")

### 9.2 XGBoost

**How it works (Gradient Boosting):** XGBoost builds trees **sequentially** — each new tree corrects the residual errors of all previous trees. The gradient of the loss function guides where to focus.

**Key advantages over RF:**
- Explicit regularisation (`reg_alpha`, `reg_lambda`) prevents overfitting
- Column sub-sampling and row sub-sampling add robustness
- `hist` tree method enables fast binned feature splits

### 9.3 LightGBM

**How it works (Leaf-wise growth):** Unlike XGBoost which grows level-by-level, LightGBM splits the **leaf with maximum delta loss** at each step. This allows deeper, more accurate trees with fewer total leaves.

**Advantages:**
- 5–10× faster than XGBoost on large datasets
- Better accuracy on tabular data due to leaf-wise growth
- Handles high-cardinality categoricals natively

**Key parameter:** `num_leaves` controls model capacity. More leaves = more complex = risk of overfitting.

In [ ]:
# Load best params for display
with open('models/best_params.json') as f:
    best_params = json.load(f)

print("📋 Model Hyperparameters (from training pipeline)")
print("\n── Random Forest ──")
rf_display = {k: v for k, v in best_params['random_forest'].items() 
              if k in ['n_estimators','max_depth','min_samples_split','min_samples_leaf','max_features']}
for k, v in rf_display.items():
    print(f"   {k}: {v}")

print("\n── XGBoost ──")
xgb_display = {k: v for k, v in best_params['xgboost'].items()
               if k in ['n_estimators','max_depth','learning_rate','subsample','colsample_bytree']}
for k, v in xgb_display.items():
    print(f"   {k}: {v}")

print("\n── LightGBM ──")
lgbm_display = {k: v for k, v in best_params['lightgbm'].items()
                if k in ['n_estimators','num_leaves','max_depth','learning_rate','subsample','colsample_bytree']}
for k, v in lgbm_display.items():
    print(f"   {k}: {v}")

---
## 10. Hyperparameter Tuning

### What is Optuna?

**Optuna** is an automatic hyperparameter optimisation framework that uses **Bayesian Optimisation** (via the Tree-structured Parzen Estimator, or TPE) to intelligently sample the parameter space.

### Why Bayesian Optimisation over Grid Search?

| Method | How it works | Trials needed |
|---|---|---|
| **Grid Search** | Exhaustive cartesian product | Exponential (curse of dimensionality) |
| **Random Search** | Uniform random sampling | Better, but still uninformed |
| **Bayesian Optimisation** | Builds a probabilistic surrogate model of the objective; samples where improvement is expected | Efficient (typically 50–200 trials) |

Bayesian optimisation uses previous trial results to form a **prior belief** about which parameter regions are promising, then focuses future trials there.

### Tuning Strategy

- **Objective function:** Maximise R² on the validation set
- **RF trials:** 60 | **XGBoost trials:** 80 | **LightGBM trials:** 80
- **Ensemble weights** tuned via fine-grained grid search (0.05 steps) on validation predictions

In [ ]:
# Load model registry to display tuning results
with open('models/model_registry.json') as f:
    registry = json.load(f)

mode_str = "Fast Mode (fixed params)" if registry.get('fast_mode') else "Full Optuna HPO"
print(f"Training mode : {mode_str}")
print(f"Last trained  : {registry.get('last_trained', 'N/A')}")
print()

print("🏆 Best Hyperparameters:")
print(f"  RF   → n_estimators={best_params['random_forest']['n_estimators']}, max_depth={best_params['random_forest']['max_depth']}")
print(f"  XGB  → n_estimators={best_params['xgboost']['n_estimators']}, max_depth={best_params['xgboost']['max_depth']}, lr={best_params['xgboost']['learning_rate']}")
print(f"  LGBM → n_estimators={best_params['lightgbm']['n_estimators']}, num_leaves={best_params['lightgbm']['num_leaves']}, lr={best_params['lightgbm']['learning_rate']}")
print()

val_m = registry.get('val_metrics', {})
print("📊 Validation R² Scores:")
print(f"  Random Forest : {val_m.get('rf',{}).get('r2', 'N/A'):.4f}")
print(f"  XGBoost       : {val_m.get('xgb',{}).get('r2', 'N/A'):.4f}")
print(f"  LightGBM      : {val_m.get('lgbm',{}).get('r2', 'N/A'):.4f}")
print(f"  Ensemble      : {val_m.get('ensemble',{}).get('r2', 'N/A'):.4f}")

---
## 11. Ensemble Learning

### Why Ensembles?

Individual models have different **error characteristics** — they make mistakes in different places. A weighted ensemble exploits this **diversity**:

- Where RF is uncertain (high variance), XGBoost or LightGBM may be confident.
- Averaging reduces variance without increasing bias.

**Result:** The ensemble typically outperforms any individual model on the validation and test sets.

### Architecture

```
       ┌─────────────────┐
       │   Input (X)     │
       └────────┬────────┘
          ┌─────┴─────┐──────────────┐
          ▼           ▼              ▼
   ┌─────────────┐ ┌──────────┐ ┌──────────┐
   │ Random      │ │ XGBoost  │ │ LightGBM │
   │ Forest      │ │          │ │          │
   └──────┬──────┘ └────┬─────┘ └────┬─────┘
    w_rf  │       w_xgb │     w_lgbm │
          └─────────────┼────────────┘
                        ▼
               ┌─────────────────┐
               │ Weighted Average │
               │ ŷ = Σ wᵢ · ŷᵢ  │
               └─────────────────┘
```

In [ ]:
ens_weights = registry.get('ensemble_weights', {})
w_rf   = ens_weights.get('rf',   0.0)
w_xgb  = ens_weights.get('xgb',  0.45)
w_lgbm = ens_weights.get('lgbm', 0.55)

print("⚖️  Optimised Ensemble Weights:")
print(f"   Random Forest : {w_rf:.0%}")
print(f"   XGBoost       : {w_xgb:.0%}")
print(f"   LightGBM      : {w_lgbm:.0%}")

# Visualise weights
fig, ax = plt.subplots(figsize=(6, 4))
model_names  = ['Random Forest', 'XGBoost', 'LightGBM']
weight_values = [w_rf, w_xgb, w_lgbm]
colors = [PRIMARY, SECONDARY, ACCENT]
bars = ax.bar(model_names, weight_values, color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, weight_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.0%}', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylim(0, max(weight_values) * 1.25)
ax.set_title('Optimised Ensemble Weights')
ax.set_ylabel('Weight')
plt.tight_layout()
plt.show()